# Navigating 3DV 2026 with Vision-Language Models and Embeddings

**3DV 2026** is the 13th edition of the International Conference on 3D Vision, held in **Vancouver, March 20–23**. The conference received a record **404 valid submissions** this year, of which **177 papers were accepted** — 22 for oral presentation and 155 for poster presentation — for an overall acceptance rate of 43.8%.

The program runs as a single-track conference over 3.5 days: 6 oral sessions (12-minute talks), 6 poster sessions (~1.5 hours each), and 4 keynote talks from:


## The Problem

Even armed with the conference schedule, deciding *which* of 177 papers to prioritise is non-trivial. Papers are distributed across 6 poster sessions and 12 oral slots, many running concurrently with talks you also want to see. A quick scan of titles is not enough — you need to understand the research landscape, spot the most distinctive work, and know where the demos and code are.

## The Approach

We load all 177 papers — full PDF page images plus structured metadata — into a [FiftyOne](https://docs.voxel51.com/) grouped dataset, then run three complementary analyses:

1. **VLM-based annotation**: A reasoning vision-language model reads the first page of each paper and extracts topic labels, author affiliations, and project page links

2. **Document embeddings**: Jina v4 encodes every page as a dense vector; per-paper embeddings are computed by mean pooling, with reference pages excluded

3. **Brain analysis**: UMAP visualizations, similarity indexes, representativeness scores, and uniqueness scores surface the papers most worth your time

The result is an interactive dataset you can explore in the FiftyOne App — filter by topic, sort by uniqueness, drill into any paper's full page sequence, and build your personal conference schedule.

In [ ]:
import fiftyone as fo

dataset = fo.load_dataset("3dvs2026_papers")

page_1=dataset.select_group_slices(slices="page_01")

## The Dataset — 177 Papers as a Grouped FiftyOne Dataset

Each paper's PDF was converted to per-page PNG images. The full set — 3,019 images across 176 papers (one submission had no available PDFs) — is loaded into a [FiftyOne grouped dataset](https://docs.voxel51.com/user_guide/groups.html) where:

- Each **group** is one paper, identified by its submission ID
- Each **slice** within a group is one page image: `page_01`, `page_02`, … for main PDF pages and `supp_page_01`, `supp_page_02`, … for supplementary material

Groups are unbalanced by design — papers with 8 pages have 8 slices, papers with 22 pages have 22. Papers without supplementary material simply have no `supp_page_*` slices. FiftyOne handles this natively.

Every sample (page) carries the full paper metadata as fields: `title`, `authors`, `abstract`, `affiliations`, `poster_session`, `oral_session`, `oral_date`, `oral_start_time`, `poster_date`, `poster_start_time`, `pdf_link`, and `supplementary_link`.

Most annotation and analysis work is done on the `page_01` slice — the cover page containing the title, author list, affiliations, abstract, and teaser figure. This is the richest single page for both VLM-based classification and embedding-based similarity.

## Using a Vision-Language Model to Annotate Papers at Scale

Reading 177 abstracts is feasible. Reading 177 abstracts *and* making considered decisions about which papers to prioritise across 6 poster sessions and 12 oral slots — while also attending talks and having hallway conversations — is not.

Instead, we use [Qwen3.5-9B](https://huggingface.co/Qwen/Qwen3.5-9B), a reasoning vision-language model, applied directly to the first page image of each paper via the [FiftyOne Model Zoo](https://docs.voxel51.com/model_zoo/index.html). The first page is the information-dense page: title, authors, affiliations, abstract, and usually a teaser figure. Running the model on this single page per paper is both efficient and effective.

Three annotation passes are run using [`apply_model`](https://docs.voxel51.com/api/fiftyone.core.collections.html#fiftyone.core.collections.SampleCollection.apply_model):

1. **Topic classification** — assign each paper to one of 10 research areas

2. **Affiliation extraction** — extract the list of author institutions

3. **Project page detection** — find links to code, demos, or supplementary sites

Qwen3.5 is a *reasoning* model: it wraps its chain-of-thought in `<think>...</think>` tags before giving its final answer. We strip the thinking tokens during parsing and keep only the structured output.

In [ ]:
import fiftyone as fo
import fiftyone.zoo as foz

# Register and download
foz.register_zoo_model_source(
    "https://github.com/harpreetsahota204/qwen3_5_vl",
    overwrite=True
)

model = foz.load_zoo_model(
    "Qwen/Qwen3.5-9B",
    media_type="image",
)

model.max_new_tokens = 32768

## Step 1 — Topic Classification

With 177 papers spanning a wide range of 3D vision subfields, a flat list of titles is hard to navigate. We give the VLM the first page of each paper and ask it to assign exactly one label from a predefined taxonomy of 10 topic areas — from *3D Reconstruction and Novel View Synthesis* through to *Novel Sensors and Specialized Domains*.

The model uses the `classify` operation mode and returns a structured JSON response (`[{"label": "topic_name"}]`), which FiftyOne stores as a [`Classification`](https://docs.voxel51.com/user_guide/using_datasets.html#classification) label. This gives us an immediately filterable and groupable taxonomy directly in the [FiftyOne App](https://docs.voxel51.com/user_guide/app.html).

In [ ]:
model.operation = "classify"

topics = [
    "3D Reconstruction and Novel View Synthesis",
    "3D Generation and Editing",
    "Human Avatars and Motion",
    "Hand-Object and Human-Scene Interaction",
    "Semantic 3D Understanding",
    "Autonomous Driving & Outdoor Perception",
    "Depth and Stereo Estimation",
    "Dynamic and 4D Scenes",
    "Geometric Vision Fundamentals",
    "Novel Sensors and Specialized Domains",
]

topics_str = "\n".join(f"- {t}" for t in topics)

prompt = (
    "You are classifying papers accepted to the International Conference on 3D Vision (3DV). "
    "You are given an image of the first page of a paper. "
    "Based on the title, abstract, and any other visible content, "
    "assign the paper to EXACTLY one of the following topics: "
    f"[{topics_str}]. "
    "Choose EXACTLY one of the topic name, exactly as shown above. Respond in this format: "
    '[{"label": "topic_name"}]. '
)

model.prompt = prompt

In [ ]:
page_1.apply_model(
    model,
    label_field="topic",
    batch_size=16,
    num_workers=4
)

## Step 2 — Extracting Author Affiliations

Knowing which institutions are most active at 3DV — and which papers come from which labs — is useful context for navigating the conference. The affiliations appear on `page_01` directly below the author names.

The model returns a Python list of institution strings, one entry per affiliation. ASCII normalisation is requested (`ü → u`, `é → e`, `ß → ss`) to avoid encoding inconsistencies when filtering or grouping by institution in the FiftyOne App. The parsed list is stored as `affiliations` on the `page_01` samples, and a derived `number_of_authors` integer field is written for quick filtering.

In [ ]:
model.operation="vqa"

model.prompt = (
    "Extract the institution affiliations associated with this academic paper. "
    "This information typically follows the title and list of author names. "
    "Each affiliation should be its own entry."
    "Use standard ASCII characters — replace accented or special characters with their plain ASCII equivalents (e.g. ü → u, é → e, ß → ss). "
    "Don't think too much, its an easy task to do."
    "Report your response as a properly formatted Python list of strings."
)

page_1.apply_model(
    model,
    label_field="affiliations",
    batch_size=16,
    num_workers=4
)

## Step 3 — Finding Project Pages and Code

A paper with a live project page or GitHub repository is more accessible than one without — it often means working code, pre-trained models, or an interactive demo you can visit at the poster. This is a useful signal for prioritising which posters to spend time at.

The model is given the first page image and asked to report any project page or repository link it finds, delimited by triple backticks. If no link is present it says so in prose, which parses to `None`. The parsed result is stored as `project_page` (a URL string or `None`) and a derived boolean `has_code` (`"Yes"` / `"No"`) is written for easy filtering in the app.

In [ ]:
model.prompt = (
    "Determine if this paper provides a link to their project page, GitHub repository, "
    "or site where additional material can be accessed. "
    "Report this link delimited by triple backticks. "
)

page_1.apply_model(
    model,
    label_field="additional_material",
    batch_size=16,
    num_workers=4
)

In [ ]:
model.operation="vqa"

model.prompt = (
    "You are given an image of the first page of a paper accepted to the International Conference on 3D Vision (3DV). "
    "Based on the title, abstract, and any other content, classify this paper by returning a JSON object "
    "with the following boolean fields: introduces_dataset, introduces_model, introduces_method, has_benchmark. "
    "For example: "
    '{"introduces_dataset": true, "introduces_model": false, "introduces_method": true, "introduces_benchmark": false}. '
    "Respond with only the JSON object, no additional text."
    "Don't think too hard, it should be clear what this paper introduces based on the information you see."
)

page_1.apply_model(
    model,
    label_field="contributions",
    batch_size=16,
    num_workers=4
)


## Parsing Model Responses

Qwen3.5 is a reasoning model — before giving its final answer it emits a chain-of-thought block wrapped in `<think>...</think>` tags. We discard everything inside those tags and parse only what follows.

Two utility functions handle the two response formats we use:

- **`parse_model_list_response`** — strips the `<think>` block via `rpartition("</think>")`, then uses a greedy regex `\[.*\]` with `re.DOTALL` to find the outermost list literal and evaluates it with `ast.literal_eval`. Works whether the model wrapped the list in a markdown code fence, added prose, or returned it bare.

- **`parse_model_url_response`** — same `<think>` stripping, then finds content between triple backticks and validates it starts with `http`. Returns `None` if no valid URL is present, so the field stays `None` rather than storing a garbage string.

Using `rpartition` (not `partition`) handles the occasional case where the model emits multiple `</think>` blocks — we always split on the *last* one so the final answer is cleanly isolated. When no `</think>` token is present at all (non-reasoning models), `rpartition` returns the full response as the third element, so the same functions work transparently on any model.

In [ ]:
import ast
import re
import json

def parse_model_list_response(response: str) -> list[str]:
    """Parse a reasoning model response into a list of strings.

    Discards everything inside <think>...</think>, then finds the first
    [...] block in the remaining text and evaluates it as a Python literal.
    Works regardless of whether the model wrapped the list in a code fence,
    added prose around it, or returned it bare. If no </think> token is
    present the entire response is searched, so non-reasoning models are
    handled transparently.
    """
    # rpartition returns ("", "", response) when the token is absent,
    # so after_think is always the text we want to search.
    _, _, after_think = response.rpartition("</think>")

    # Greedy match from the first '[' to the last ']', spanning newlines
    match = re.search(r"\[.*\]", after_think, re.DOTALL)
    if not match:
        raise ValueError(f"No list literal found in response:\n{after_think!r}")

    return ast.literal_eval(match.group())


def parse_model_url_response(response: str) -> str | None:
    """Parse a reasoning model response into a URL string, or None.

    The model is asked to delimit its answer with triple backticks.
    Returns the URL if one is found inside the backticks, or None when
    the model indicated no link exists (e.g. content like 'here' or
    'Not provided in the image').
    """
    _, _, after_think = response.rpartition("</think>")

    # Extract all content between triple backtick pairs
    match = re.search(r"```(.*?)```", after_think, re.DOTALL)
    if not match:
        return None

    candidate = match.group(1).strip()

    # Only return the value if it looks like a real URL
    return candidate if candidate.startswith("http") else None


def parse_model_json_response(response: str) -> dict[str, str]:
    """Parse a reasoning model response into a dict of Yes/No strings.

    Discards everything inside <think>...</think>, then finds the first
    {...} block in the remaining text and parses it as JSON. Boolean
    values are normalized to "Yes"/"No" strings, handling true/True/false/False
    and quoted variants. If no </think> token is present the entire response
    is searched, so non-reasoning models are handled transparently.
    """
    _, _, after_think = response.rpartition("</think>")

    match = re.search(r"\{.*\}", after_think, re.DOTALL)
    if not match:
        raise ValueError(f"No JSON object found in response:\n{after_think!r}")

    # Normalize unquoted Python-style booleans to JSON-compatible lowercase
    raw = match.group()
    raw = re.sub(r"\bTrue\b", "true", raw)
    raw = re.sub(r"\bFalse\b", "false", raw)

    parsed = json.loads(raw)

    return {
        k: "Yes" if v in (True, "true", "True", "yes", "Yes") else "No"
        for k, v in parsed.items()
    }



In [ ]:
affiliations = [parse_model_list_response(r) for r in dataset.values("affiliations_response")]
dataset.set_values("affiliations", affiliations)

In [ ]:
url_links = [parse_model_url_response(r) for r in dataset.values("additional_material_response")]
dataset.set_values("project_page", url_links)

In [ ]:
n_authors = [len(auth) for auth in dataset.values("authors")]
dataset.set_values("number_of_authors", n_authors)

In [ ]:
# Compute values externally, then write them
has_code = ["Yes" if url is not None else "No" for url in dataset.values("project_page")]
dataset.set_values("has_code", has_code)

In [ ]:

fields = ["introduces_dataset", "introduces_model", "introduces_method", "introduces_benchmark"]
responses = dataset.values("contributions_response")
parsed = [parse_model_json_response(r) for r in responses]

for field in fields:
    dataset.set_values(field, [p.get(field, "No") for p in parsed])


In [ ]:
# # Iterate over all slices EXCEPT the one you want to keep
# slices_to_modify = [s for s in dataset.group_slices if s != "page_01"]

# for slice_name in slices_to_modify:
#     # Get a view of just this slice
#     slice_view = dataset.select_group_slices([slice_name])
    
#     # Iterate and delete the field from samples in this slice
#     for sample in slice_view.iter_samples(autosave=True):
#         if "affiliations" in sample.field_names:
#             sample.clear_field("affiliations")

In [ ]:
# for slice_name in dataset.group_slices:
#     view = dataset.select_group_slices([slice_name])
#     view.set_values("affiliations", affiliations)
#     view.set_values("project_page", url_links)


## Embedding Every Page for Visual Similarity Search

The VLM annotations give us structured labels, but to find papers that are *visually and semantically similar* — regardless of how they were labelled — we need dense vector embeddings.

We use [Jina Embeddings v4](https://huggingface.co/jinaai/jina-embeddings-v4) (`jinaai/jina-embeddings-v4`, retrieval task), a document embedding model that encodes images as 2048-dim vectors optimised for retrieval. Every page across all 105 slices is embedded — 3,019 vectors in total — using [`compute_embeddings`](https://docs.voxel51.com/api/fiftyone.core.collections.html#fiftyone.core.collections.SampleCollection.compute_embeddings) on the fully flattened view.

From these embeddings we build two [FiftyOne Brain](https://docs.voxel51.com/brain.html) artifacts:

- **UMAP visualization** ([`fob.compute_visualization`](https://docs.voxel51.com/api/fiftyone.brain.html#fiftyone.brain.compute_visualization)) — projects the 2048-dim vectors to 2D for interactive exploration in the app. Pages from the same paper naturally cluster together, and reference pages form a visually distinct island that is easy to identify and tag.

- **Similarity index** ([`fob.compute_similarity`](https://docs.voxel51.com/api/fiftyone.brain.html#fiftyone.brain.compute_similarity)) — enables nearest-neighbour search directly in the app: click any paper page and find the most visually similar pages across the entire conference.

In [ ]:
import fiftyone as fo
import fiftyone.zoo as foz

# Load Jina v4 model with desired configuration
emb_model = foz.load_zoo_model(
    "jinaai/jina-embeddings-v4",
    task="retrieval", 
)

# Select multiple slices as a flattened view
flattened_view = dataset.select_group_slices(dataset.group_slices)

flattened_view.compute_embeddings(emb_model, embeddings_field=f"jinav4_embeddings")

In [ ]:
import fiftyone.brain as fob

fob.compute_visualization(
    flattened_view,
    embeddings="jinav4_embeddings",
    brain_key="jina_viz",
    method="umap",
    batch_size=128,
    num_workers=16,
    num_dims=2
)

import fiftyone.brain as fob
# Build a similarity index
fob.compute_similarity(
    flattened_view,
    model="jinaai/jina-embeddings-v4",
    embeddings="jinav4_embeddings",
    brain_key="jina_sim",
    batch_size=128,
    num_workers=16,
)

In [ ]:
dataset.save_view("flattened_view", flattened_view)
dataset.save()

## Paper-Level Embeddings via Mean Pooling

Per-page embeddings are useful for page-level similarity search, but for ranking and comparing *papers* we want a single fixed-size vector per paper — regardless of whether it has 8 pages or 22.

The approach: group all page embeddings by `paper_id`, exclude pages tagged `reference-page` (identified as a visual outlier cluster in the UMAP — dense citation lists carry no signal about a paper's contribution), and take the element-wise mean of the remaining pages.

This gives a 2048-dim vector for every paper that reflects the paper's overall visual and semantic content. Because the reference pages are excluded before pooling, the vector is dominated by the method, results, and figure-heavy pages where the actual contribution lives.

The pooled embedding is written to every slice of each paper (not just `page_01`) so it is available for querying across any view of the dataset. See [FiftyOne Tags](https://docs.voxel51.com/user_guide/using_datasets.html#tags) for how the `reference-page` tag was applied.

In [ ]:
import numpy as np
from collections import defaultdict

# Pull paper_id, embedding, and tags together
paper_ids, embeddings, tags_list = flattened_view.values(["paper_id", "jinav4_embeddings", "tags"])

# Pool only pages that are NOT tagged as reference-page
groups = defaultdict(list)
for pid, emb, tags in zip(paper_ids, embeddings, tags_list):
    if "reference-page" not in tags:
        groups[pid].append(emb)

# 3. Mean pool each group → one fixed-size vector per paper
pooled = {pid: np.mean(np.stack(embs), axis=0) for pid, embs in groups.items()}

# 4. Map the pooled vector back to every sample in order
pooled_per_sample = [pooled[pid] for pid in paper_ids]

# 5. Write back to the dataset
flattened_view.set_values("all_page_embeddings_pooled", pooled_per_sample)

In [ ]:
import fiftyone.brain as fob

fob.compute_visualization(
    flattened_view,
    embeddings="all_page_embeddings_pooled",
    brain_key="all_page_embeddings_viz",
    method="umap",
    batch_size=128,
    num_workers=16,
    num_dims=2
)

import fiftyone.brain as fob
# Build a similarity index
fob.compute_similarity(
    flattened_view,
    model="jinaai/jina-embeddings-v4",
    embeddings="all_page_embeddings_pooled",
    brain_key="all_page_embeddings_sim",
    batch_size=128,
    num_workers=16,
)

## Ranking Papers — Representativeness and Uniqueness

With per-paper embeddings in place we can score every paper along two complementary axes using the [FiftyOne Brain](https://docs.voxel51.com/brain.html):

**Representativeness** ([`fob.compute_representativeness`](https://docs.voxel51.com/api/fiftyone.brain.html#fiftyone.brain.compute_representativeness))
Scores each paper by proximity to the nearest cluster centre in embedding space. A high score means the paper is an archetypal example of its research area — the kind of work that defines what a topic is *about* at this edition of the conference.

**Uniqueness** ([`fob.compute_uniqueness`](https://docs.voxel51.com/api/fiftyone.brain.html#fiftyone.brain.compute_uniqueness))
Scores each paper by how far it sits from its nearest neighbours. A high score means the paper occupies its own region of the embedding space — it is doing something that few or no other papers at 3DV are doing.

Together these two scores give you a principled shortlist strategy: sort by representativeness to understand the shape of the conference; sort by uniqueness to find the outliers most likely to show you something genuinely new.

In [ ]:
import fiftyone.brain as fob

# Compute representativeness scores
fob.compute_representativeness(
    flattened_view,
    representativeness_field="jina_represent",
    method="cluster-center",
    embeddings="all_page_embeddings_pooled"
)

# Find most representative samples
representative_view = dataset.sort_by("jina_represent", reverse=True)

# Detect duplicates using embeddings
results = fob.compute_uniqueness(
    flattened_view,
    uniqueness_field="jina_uniqueness",
    embeddings="all_page_embeddings_pooled"
)

# Filter to most unique samples
unique_view = dataset.sort_by("uniqueness", reverse=True)

## What to See at 3DV 2026

The dataset is now fully annotated. Here's how to use it to plan your conference days:

**Finding your topic areas**
Open the [FiftyOne App](https://docs.voxel51.com/user_guide/app.html), load the `page_01` view, and filter by `topic` to browse papers in a subfield you care about. The UMAP (`all_page_embeddings_viz`) will show you how papers cluster — papers sitting close together share visual and semantic similarity even across topic labels.

**Surfacing the most distinctive work**
Sort `page_01` by `jina_uniqueness` descending. High-uniqueness papers are the ones doing something genuinely different from the rest of the conference — these are worth a closer look regardless of topic.

**Finding archetypal work in each area**
Sort by `jina_represent` descending within a topic to find the papers that best represent the centre of each research cluster — useful for getting up to speed on a subfield quickly.

**Building your poster shortlist**
Filter by `has_code == "Yes"` for papers with a live project page, then sort by uniqueness. These are likely to have working demos at their poster boards.

**Scheduling around orals**
All oral papers have `oral_date` and `oral_start_time` fields populated. Cross-reference your shortlist against the schedule — oral sessions run for one hour with 12-minute talks, and every oral paper also appears in a poster session the same day.

With 22 orals across 6 oral sessions and 155 posters across 6 poster sessions over 3.5 days, this dataset gives you a systematic way to cut through the noise and make the most of your time in Vancouver.